# MMF GPU Model Runner
Runs a single GPU model (global or foundation). Called by orchestrator notebooks via `dbutils.notebook.run()`.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("model", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("use_case", "")
dbutils.widgets.text("train_table", "")
dbutils.widgets.text("freq", "")
dbutils.widgets.text("prediction_length", "")
dbutils.widgets.text("backtest_length", "")
dbutils.widgets.text("stride", "")
dbutils.widgets.text("metric", "smape")
dbutils.widgets.text("group_id", "unique_id")
dbutils.widgets.text("date_col", "ds")
dbutils.widgets.text("target", "y")
dbutils.widgets.text("num_nodes", "1")

In [ ]:
model_class = "global" if "NeuralForecast" in dbutils.widgets.get("model") else "foundation"
%pip install "mmf_sa[{model_class}] @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git@v0.1.2" --quiet
dbutils.library.restartPython()

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model = dbutils.widgets.get("model")
run_id = dbutils.widgets.get("run_id")
use_case = dbutils.widgets.get("use_case")
train_table = dbutils.widgets.get("train_table")
freq = dbutils.widgets.get("freq")
prediction_length = int(dbutils.widgets.get("prediction_length"))
backtest_length = int(dbutils.widgets.get("backtest_length"))
stride = int(dbutils.widgets.get("stride"))
metric = dbutils.widgets.get("metric")
group_id = dbutils.widgets.get("group_id")
date_col = dbutils.widgets.get("date_col")
target = dbutils.widgets.get("target")
num_nodes = int(dbutils.widgets.get("num_nodes"))

In [ ]:
from mmf_sa import run_forecast

user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]

run_forecast(
    spark=spark,
    train_data=f"{catalog}.{schema}.{train_table}",
    scoring_data=f"{catalog}.{schema}.{train_table}",
    scoring_output=f"{catalog}.{schema}.{use_case}_scoring_output",
    evaluation_output=f"{catalog}.{schema}.{use_case}_evaluation_output",
    model_output=f"{catalog}.{schema}",
    group_id=group_id,
    date_col=date_col,
    target=target,
    freq=freq,
    prediction_length=prediction_length,
    backtest_length=backtest_length,
    stride=stride,
    metric=metric,
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=[model],
    experiment_path=f"/Users/{user}/mmf/{use_case}",
    use_case_name=use_case,
    run_id=run_id,
    accelerator="gpu",
    num_nodes=num_nodes,
)